In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json


In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\train"
VAL_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\val"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\test"

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes

Found 2351 images belonging to 3 classes.
Found 506 images belonging to 3 classes.
Found 505 images belonging to 3 classes.


In [12]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [13]:
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 2048)                │           8,192 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,121,219 (92.02 MB)

 Trainable params: 529,411 (2.02 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

In [14]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "ResNet50_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [15]:
history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5508 - loss: 1.1806
Epoch 1: val_accuracy improved from -inf to 0.82806, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 184s 2s/step - accuracy: 0.5524 - loss: 1.1768 - val_accuracy: 0.8281 - val_loss: 0.5270 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8043 - loss: 0.5448
Epoch 2: val_accuracy improved from 0.82806 to 0.85771, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 0.8043 - loss: 0.5450 - val_accuracy: 0.8577 - val_loss: 0.3861 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8224 - loss: 0.4815
Epoch 3: val_accuracy improved from 0.85771 to 0.87352, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.8224 - loss: 0.4813 - val_accuracy: 0.8735 - val_loss: 0.3313 - learning_rate: 1.0000e-04
Epoch 4/50
74/74 ━━

In [16]:


with open("resnet50_head_history_LWDCD.json", "w") as f:
    json.dump(history_head.history, f)

In [17]:
base_model.trainable = True

for layer in base_model.layers[:120]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9152 - loss: 0.2103
Epoch 1: val_accuracy did not improve from 0.94862
74/74 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.9152 - loss: 0.2103 - val_accuracy: 0.9466 - val_loss: 0.1298 - learning_rate: 1.0000e-05
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9398 - loss: 0.1654
Epoch 2: val_accuracy improved from 0.94862 to 0.96245, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 160s 2s/step - accuracy: 0.9398 - loss: 0.1654 - val_accuracy: 0.9625 - val_loss: 0.1117 - learning_rate: 1.0000e-05
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9486 - loss: 0.1452
Epoch 3: val_accuracy improved from 0.96245 to 0.96443, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - accuracy: 0.9485 - loss: 0.1453 - val_accuracy: 0.9644 - val_loss: 0.1048 - learning_rate: 1.0000e-05
Epoch 4/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.940

In [18]:
with open("resnet50_finetune_history_LWDCD.json", "w") as f:
    json.dump(history_fine.history, f)

In [19]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

16/16 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.9673 - loss: 0.1550
Test accuracy: 0.9663


In [20]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step
                  precision    recall  f1-score   support

   Healthy Wheat       0.97      0.97      0.97       175
       Leaf Rust       0.98      0.96      0.97       190
Wheat Loose Smut       0.94      0.98      0.96       140

        accuracy                           0.97       505
       macro avg       0.96      0.97      0.97       505
    weighted avg       0.97      0.97      0.97       505

[[169   1   5]
 [  4 182   4]
 [  1   2 137]]


In [21]:
model.save("ResNet50_finetuned_LWDCD.keras")

In [22]:
model.save("ResNet50_head_LWDCD.keras")

In [23]:
print("Training class indices:", train_generator.class_indices)
print("Validation class indices:", val_generator.class_indices)
print("Original test class indices:", test_generator.class_indices)

Training class indices: {'Healthy Wheat': 0, 'Leaf Rust': 1, 'Wheat Loose Smut': 2}
Validation class indices: {'Healthy Wheat': 0, 'Leaf Rust': 1, 'Wheat Loose Smut': 2}
Original test class indices: {'Healthy Wheat': 0, 'Leaf Rust': 1, 'Wheat Loose Smut': 2}


In [5]:
import json

class_indices = train_generator.class_indices

# Convert to ordered list
class_names = [None] * len(class_indices)
for name, idx in class_indices.items():
    class_names[idx] = name

with open("class_names_LWDC.json", "w") as f:
    json.dump(class_names, f, indent=4)